Data PreProcessing


In [2]:
import pandas as pd
import os
from pathlib import Path

# Load data
project_root = Path(os.getcwd()).resolve().parent
data_path = project_root / "data" / "raw" / "crime_dataset_india.csv"

df = pd.read_csv(data_path)

# Basic cleaning
df.columns = df.columns.str.strip()
df["City"] = df["City"].str.strip().str.title()  # Normalize city names

# Crime count per city
crime_city = df["City"].value_counts().reset_index()
crime_city.columns = ["City", "Crime Count"]

print(crime_city.head())

        City  Crime Count
0      Delhi         5400
1     Mumbai         4415
2  Bangalore         3588
3  Hyderabad         2881
4    Kolkata         2518


In [ ]:
import pandas as pd
from pathlib import Path

def preprocess_crime_data(raw_path, save_path):
    # Load raw data
    df = pd.read_csv(raw_path)

    # Convert dates & extract year/month
    df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], errors='coerce')
    df = df.dropna(subset=['Date of Occurrence'])
    df['Year'] = df['Date of Occurrence'].dt.year
    df['Month'] = df['Date of Occurrence'].dt.month

    # Normalize city names
    df['City'] = df['City'].str.strip().str.title()

    # City to State mapping (add or adjust if needed)
    city_to_state = {
        "Ahmedabad": "Gujarat",
        "Chennai": "Tamil Nadu",
        "Ludhiana": "Punjab",
        "Pune": "Maharashtra",
        "Delhi": "Delhi",
        "Mumbai": "Maharashtra",
        "Bangalore": "Karnataka",
        "Hyderabad": "Telangana",
        "Kolkata": "West Bengal",
        "Jaipur": "Rajasthan",
        "Lucknow": "Uttar Pradesh",
        "Kanpur": "Uttar Pradesh",
        "Surat": "Gujarat",
        "Nagpur": "Maharashtra",
        "Agra": "Uttar Pradesh",
        "Visakhapatnam": "Andhra Pradesh",
        "Thane": "Maharashtra",
        "Ghaziabad": "Uttar Pradesh",
        "Indore": "Madhya Pradesh",
        "Patna": "Bihar",
        "Bhopal": "Madhya Pradesh",
        "Meerut": "Uttar Pradesh",
        "Srinagar": "Jammu & Kashmir",
        "Nashik": "Maharashtra",
        "Vasai": "Maharashtra",
        "Varanasi": "Uttar Pradesh",
        "Kalyan": "Maharashtra",
        "Faridabad": "Haryana",
        "Rajkot": "Gujarat",
    }
    df['State'] = df['City'].map(city_to_state)

    # Drop rows with unmapped states
    df = df.dropna(subset=['State'])

    # Aggregate: count crimes by State, Year, Month, Crime Description
    agg_df = (
        df.groupby(['State', 'Year', 'Month', 'Crime Description'])
        .size()
        .reset_index(name='Crime Count')
    )

    # Save processed data for visualization
    save_path.parent.mkdir(parents=True, exist_ok=True)
    agg_df.to_csv(save_path, index=False)

    print(f"Processed data saved to {save_path}")


raw_path = project_root / "data" / "raw" / "crime_dataset_india.csv"
save_path = project_root / "data" / "processed" / "crime_data_processed.csv"

preprocess_crime_data(raw_path, save_path)


Visualization Testing

In [ ]:
import json
import pandas as pd
import plotly.express as px

# Load the aggregated data
prepared_data_set = project_root / "data" / "processed" / "crime_data_processed.csv"
df = pd.read_csv(prepared_data_set)


geojson_loc = project_root / "json" / "cleaned.geojson"
with open(geojson_loc) as f:
    india_states = json.load(f)

all_states = [feature["properties"]["NAME_1"] for feature in india_states["features"]]
all_states_df = pd.DataFrame({"State": all_states})

# print(india_states['features'][0]['properties'])
# Filter data example

year_filter = 2020
month_filter = 1
crime_filter = 'BURGLARY'

df_filtered = df[
    (df['Year'] == year_filter) &
    (df['Month'] == month_filter) &
    (df['Crime Description'] == crime_filter)
]

merged_df = all_states_df.merge(df_filtered, on="State", how="left").fillna(0)
merged_df["Crime Count"] = merged_df["Crime Count"].astype(int)

fig = px.choropleth(
    merged_df,              # Loading data
    geojson=india_states,   # Loading India states geojson
    locations='State',
    featureidkey='properties.NAME_1',       
    color='Crime Count',
    color_continuous_scale='Reds',
    scope='asia',
    title=f'{crime_filter} Cases in {year_filter}-{month_filter:02d}'
)

fig.update_geos(fitbounds="locations", visible=False)
fig.show()


<>:6: SyntaxWarning:

invalid escape sequence '\C'

<>:8: SyntaxWarning:

invalid escape sequence '\C'

<>:6: SyntaxWarning:

invalid escape sequence '\C'

<>:8: SyntaxWarning:

invalid escape sequence '\C'

C:\Users\Hariv\AppData\Local\Temp\ipykernel_12228\2743477477.py:6: SyntaxWarning:

invalid escape sequence '\C'

C:\Users\Hariv\AppData\Local\Temp\ipykernel_12228\2743477477.py:8: SyntaxWarning:

invalid escape sequence '\C'

